# LAB 3
### PUBH 4201 Lab 3

##### Today we are using regex to parse through messy data. We are going to take our mess_data_csv and turn it into a tibble with standardization

In [17]:
import pandas as pd
import numpy as np
clean_our_data = pd.read_csv('messy_samples.csv')

##### Our data isn't exactly the clearest. Let's use some regex coding to fix it

In [8]:
# We've read our data into a data frame. Lets check our sex variable
clean_our_data['sex'].value_counts()


sex
Male       10
F           9
unknown     9
m           6
f           6
M           6
Female      5
U           4
Name: count, dtype: int64

#### Thats pretty confusing! We want to simplify the sexes into male, female, and other!

In [18]:
# Match the whole value so "female" cannot accidentally match "male".
sex = clean_our_data['sex'].astype('string').str.strip()
clean_our_data['sex'] = np.select(
    [
        sex.str.fullmatch(r'(?:m|male)', case=False, na=False),
        sex.str.fullmatch(r'(?:f|female)', case=False, na=False),
    ],
    ['Male', 'Female'],
    default='Unknown',
)

# Keep the original CSV unchanged and save the standardized data separately.
clean_our_data.to_csv('clean_our_data.csv', index=True)
clean_our_data['sex'].value_counts()

sex
Male       22
Female     20
Unknown    18
Name: count, dtype: int64

#### Great! now our data is easy to parse. Now, there are still more variables that are messy. Lets fix those too

In [10]:
# Lets check out the dob variable in our dataset
clean_our_data['dob'].value_counts()

dob
07/09/1962     1
12/16/1974     1
11.24.53       1
1957-02-27     1
1997-12-02     1
05-Dec-1968    1
1969-09-04     1
08/20/1999     1
07/19/1995     1
10/01/1951     1
07.09.61       1
06.07.96       1
26-Oct-1952    1
09.12.58       1
02.11.98       1
1958-05-15     1
07.16.15       1
1957-03-10     1
16-Sep-2000    1
07/05/2000     1
11/16/2004     1
25-Nov-2006    1
06.30.99       1
31-Jul-1953    1
11.19.65       1
06.22.53       1
24-Mar-1956    1
26-Dec-1988    1
04-Aug-2009    1
1984-01-06     1
11.11.95       1
08/17/1992     1
06.07.12       1
04.15.95       1
13-Dec-1956    1
04/21/1987     1
2015-11-28     1
12/02/1997     1
04.12.80       1
06/21/2008     1
01/20/1965     1
2006-04-10     1
02.19.51       1
04.13.98       1
01-Jan-1984    1
02/21/1967     1
21-Jun-1977    1
04/23/1978     1
05-Feb-1966    1
02-Jul-2001    1
08/02/2011     1
20-Sep-1989    1
07/10/1999     1
2006-01-28     1
14-Oct-2011    1
06/30/2010     1
20-Aug-1986    1
23-Aug-1972    1
06-Apr-196

In [19]:
# Standardize DOB values as MM/DD/YYYY using regex patterns
import re

month_numbers = {
    'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
    'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
    'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12',
}

def normalize_dob(value):
    if pd.isna(value):
        return value

    value = str(value).strip()
    match = re.fullmatch(r'(\d{1,2})[-/ ]+([A-Za-z]+)[-/ ,]+(\d{2,4})', value)
    if match:
        day, month_name, year = match.groups()
        month = month_numbers.get(month_name[:3].lower())
    else:
        match = re.fullmatch(r'([A-Za-z]+)[-/ ,]+(\d{1,2})[, -]+(\d{2,4})', value)
        if match:
            month_name, day, year = match.groups()
            month = month_numbers.get(month_name[:3].lower())
        else:
            match = re.fullmatch(r'(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})', value)
            if match:
                year, month, day = match.groups()
            else:
                match = re.fullmatch(r'(\d{1,2})[./-](\d{1,2})[./-](\d{2,4})', value)
                if not match:
                    return value
                month, day, year = match.groups()

    if month is None:
        return value
    if len(year) == 2:
        year = ('20' if int(year) <= 25 else '19') + year

    return f'{int(month):02d}/{int(day):02d}/{year}'

clean_our_data['dob'] = clean_our_data['dob'].apply(normalize_dob)
clean_our_data.to_csv('clean_our_data.csv', index=False)
clean_our_data['dob'].value_counts()

dob
12/02/1997    2
07/09/1962    1
12/16/1974    1
11/24/1953    1
02/27/1957    1
12/05/1968    1
09/04/1969    1
08/20/1999    1
07/19/1995    1
10/01/1951    1
07/09/1961    1
06/07/1996    1
10/26/1952    1
09/12/1958    1
02/11/1998    1
05/15/1958    1
07/16/2015    1
03/10/1957    1
09/16/2000    1
07/05/2000    1
11/16/2004    1
11/25/2006    1
06/30/1999    1
07/31/1953    1
11/19/1965    1
06/22/1953    1
03/24/1956    1
12/26/1988    1
08/04/2009    1
01/06/1984    1
11/11/1995    1
08/17/1992    1
06/07/2012    1
04/15/1995    1
12/13/1956    1
04/21/1987    1
11/28/2015    1
04/12/1980    1
06/21/2008    1
01/20/1965    1
04/10/2006    1
02/19/1951    1
04/13/1998    1
01/01/1984    1
02/21/1967    1
06/21/1977    1
04/23/1978    1
02/05/1966    1
07/02/2001    1
08/02/2011    1
09/20/1989    1
07/10/1999    1
01/28/2006    1
10/14/2011    1
06/30/2010    1
08/20/1986    1
08/23/1972    1
04/06/1968    1
02/27/1970    1
Name: count, dtype: int64

#### Beautiful! it's all coming together now. Let's check the glucose variables now

In [20]:
# Combine the glucose_value and glucose_unit columns into a single column named 'glucose'
clean_our_data['glucose'] = clean_our_data['glucose_value'].astype(str) + ' ' + clean_our_data['glucose_unit']
clean_our_data['glucose'].value_counts()

glucose
114.9 mg/dL      2
145.9 mg/dL      1
146.1 MG/DL      1
84.2 MG/DL       1
120.0 MG/DL      1
138.3 mg/dl      1
159.9 mmol/L     1
206.0 mg/dL      1
229.3 MG/DL      1
185.0 mg/dl      1
215.2 mg/dl      1
165.0 mmol/L     1
110.4 MG/DL      1
112.8 mmol/L     1
144.0 mg/dL      1
103.0 mmol/L     1
157.4 mg/dl      1
123.4 mg/dL      1
82.2 mg/dl       1
190.6 MG/DL      1
108.4 MG/DL      1
88.6 mg/dl       1
233.0 mg/dL      1
83.5 mg/dl       1
202.6 MG/DL      1
217.0 mmol/L     1
161.7 MG/DL      1
179.1 mg/dL      1
222.6 MG/DL      1
168.6 mg/dL      1
85.3 MG/DL       1
229.5 mg/dl      1
203.9 mmol/L     1
130.5 mg/dL      1
145.6 mg/dL      1
119.1 mmol/L     1
171.6 mg/dL      1
241.2* mmol/L    1
105.9 mg/dL      1
89.5 mg/dL       1
209.0* mg/dL     1
148.0 MG/DL      1
218.6 MG/DL      1
223.2 mg/dl      1
143.2 mg/dl      1
165.6 mmol/L     1
108.9 mmol/L     1
166.1 MG/DL      1
149.3 mg/dl      1
216.0 MG/DL      1
172.4 mg/dL      1
143.5 mmol/L     1
168.

#### Something is off? Some of these units are not the same as the other. To fix this, let's do some unit conversion

In [ ]:
# Convert mmol/L values in the combined glucose column to mg/dL using regex.
import re

glucose_match = clean_our_data['glucose'].astype('string').str.extract(
    r'(?i)^\s*(?P<value>\d+(?:\.\d+)?)\s*\*?\s*mmol/L\s*$'
)
mmol_mask = glucose_match['value'].notna()
converted_values = (
    pd.to_numeric(glucose_match.loc[mmol_mask, 'value']) * 18
).map(lambda value: f'{value:.1f} mg/dL')
clean_our_data.loc[mmol_mask, 'glucose'] = converted_values
clean_our_data.to_csv('clean_our_data.csv', index=False)
clean_our_data['glucose'].value_counts()

glucose
114.9 mg/dL     2
145.9 mg/dL     1
146.1 MG/DL     1
84.2 MG/DL      1
120.0 MG/DL     1
138.3 mg/dl     1
2878.2 mg/dL    1
206.0 mg/dL     1
229.3 MG/DL     1
185.0 mg/dl     1
215.2 mg/dl     1
2970.0 mg/dL    1
110.4 MG/DL     1
2030.4 mg/dL    1
144.0 mg/dL     1
1854.0 mg/dL    1
157.4 mg/dl     1
123.4 mg/dL     1
82.2 mg/dl      1
190.6 MG/DL     1
108.4 MG/DL     1
88.6 mg/dl      1
233.0 mg/dL     1
83.5 mg/dl      1
202.6 MG/DL     1
3906.0 mg/dL    1
161.7 MG/DL     1
179.1 mg/dL     1
222.6 MG/DL     1
168.6 mg/dL     1
85.3 MG/DL      1
229.5 mg/dl     1
3670.2 mg/dL    1
130.5 mg/dL     1
145.6 mg/dL     1
2143.8 mg/dL    1
171.6 mg/dL     1
4341.6 mg/dL    1
105.9 mg/dL     1
89.5 mg/dL      1
209.0* mg/dL    1
148.0 MG/DL     1
218.6 MG/DL     1
223.2 mg/dl     1
143.2 mg/dl     1
2980.8 mg/dL    1
1960.2 mg/dL    1
166.1 MG/DL     1
149.3 mg/dl     1
216.0 MG/DL     1
172.4 mg/dL     1
2583.0 mg/dL    1
168.0 mg/dl     1
2881.8 mg/dL    1
77.2 mg/dl      1
88

In [23]:
#Now, update our file by dropping the glucose_values and glucose_unit columns.
clean_our_data = clean_our_data.drop(columns=['glucose_value', 'glucose_unit'])
clean_our_data.to_csv('clean_our_data.csv', index=False)